# Pump: support bearing, sensor vs computer vision

Saskatchewan Polytechnic PROJ 613, Group 9 (Ganesh Mishra, Lakshya Sharma, Sagar Patel). **Week 4.**

| | Sensor branch | Computer-vision branch |
|---|---|---|
| Data | University of Ottawa bearing vibration V2 (accelerometer, bearing **test rig**) | XJTU-DV-Pump event camera (**laboratory pump**, bearing housing) |
| Classes | healthy, inner race, outer race, ball fault | healthy, inner race, outer race, ball fault |
| Test units | 24 held-out C/D recordings | 24 held-out 30 Hz recordings |
| Script | `Models/pump_bearing_independent_baseline.py` | `Models/pump_event_visual_baseline.py` (OpenCV HOG) |

Both branches predict the same four **present** bearing conditions and hold out whole recordings. They come from **different rigs**, so there are no paired sensor/image samples, no fusion, and the score gap does not rank one modality above the other. This is a component-level laboratory study, not a mining-pump validation, and it does not predict a repair date.

Each step runs the project's script unchanged, so this notebook always matches the code.

In [1]:
DATA = [
    ("Sensor: Ottawa vibration (48 MAT)", "Raw_Data/uottawa_bearing", "*.mat", 48),
    ("CV: XJTU-DV-Pump event camera (2 zips, 16.5 GB)", "Raw_Data/xjtu_dv_pump", "*.zip", 2),
    ("Supporting: original pump XLSX", "Raw_Data/pump_bearing", "*.xlsx", 4),
    ("Supporting: UCI hydraulic", "Raw_Data/uci_hydraulic", "*.zip", 1),
    ("Supporting: ESPset V3", "Raw_Data/espset_v3", "*.zip", 1),
]

import contextlib
import io
import json
import os
import platform
import runpy
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"


def package(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


try:
    commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "unknown"
record = {"git commit": commit, "python": platform.python_version()}
record.update({n: package(n) for n in ("numpy", "scipy", "pandas", "scikit-learn", "opencv-python", "matplotlib")})
display(pd.Series(record, name="environment").to_frame())

rows = []
for label, path, pattern, expected in DATA:
    p = ROOT / path
    files = [f for f in p.rglob(pattern) if f.is_file()] if p.is_dir() else ([p] if p.is_file() else [])
    rows.append({"data": label, "location": path, "files": f"{len(files)}/{expected}",
                 "MB": round(sum(f.stat().st_size for f in files) / 1e6, 1), "local": len(files) >= expected})
availability = pd.DataFrame(rows).set_index("data")
display(availability)
LOCAL = availability["local"].to_dict()

if all(LOCAL.values()):
    for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        os.environ[name] = "http://127.0.0.1:9"  # offline guard: any network call now fails loudly
    os.environ["NO_PROXY"] = ""
    print("All data is local: running fully offline.")
else:
    print("Some data is missing: those steps are skipped. Download routes are in Documentation/capstone_details.md.")


def run_script(relative_path, *args, tail=3):
    """Run a project script exactly as `python <script>` would and show its last output lines."""
    path, buffer, saved = ROOT / relative_path, io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")
    print(f"[{relative_path}]")
    for line in [l for l in text.splitlines() if l.strip()][-tail:]:
        print("  " + line)


def load(name):
    path = RESULTS / name
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None


def metrics(block):
    keys = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")
    return {k: round(block[k], 3) for k in keys if isinstance(block.get(k), (int, float))}


def check(before, after, paths, name):
    def pick(doc, path):
        for key in path:
            doc = doc[key]
        return metrics(doc)
    same = before is not None and all(pick(before, p) == pick(after, p) for p in paths)
    print(f"{name}: {'matches' if same else 'DIFFERS from'} the committed result")

,environment
git commit,b9e72ba
python,3.11.9
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: Ottawa vibration (48 MAT),Raw_Data/uottawa_bearing,48/48,631.5,True
"CV: XJTU-DV-Pump event camera (2 zips, 16.5 GB)",Raw_Data/xjtu_dv_pump,2/2,16488.7,True
Supporting: original pump XLSX,Raw_Data/pump_bearing,4/4,3.1,True
Supporting: UCI hydraulic,Raw_Data/uci_hydraulic,1/1,76.6,True
Supporting: ESPset V3,Raw_Data/espset_v3,1/1,0.3,True


All data is local: running fully offline.


## 1. Sensor branch: vibration

Each 10 s Ottawa recording becomes 100 windows of 0.1 s with 16 time and spectral features. A random forest trains on the 24 A/B speed-profile recordings and predicts each of the 24 separate C/D recordings once (majority vote over its windows). Controls: most-frequent class, held-out speed profile, and shuffled training labels.

In [2]:
before = load("pump_bearing_independent.json")
if LOCAL["Sensor: Ottawa vibration (48 MAT)"]:
    run_script("Models/pump_bearing_independent_baseline.py")
sensor = load("pump_bearing_independent.json")
check(before, sensor, [("random_forest_run_level",), ("held_out_speed_profile_run_level",)], "Vibration result")
display(pd.DataFrame({
    "Random forest, held-out C/D recordings (24)": metrics(sensor["random_forest_run_level"]),
    "Held-out speed profile (24)": metrics(sensor["held_out_speed_profile_run_level"]),
    "Shuffled training labels (control)": metrics(sensor["shuffled_training_run_labels_control"]),
    "Most-frequent class (control)": metrics(sensor["dummy_run_level"]),
}).T)

[Models/pump_bearing_independent_baseline.py]
  Saved Results\pump_bearing_independent.json and Figures\pump_bearing_independent_confusions.png
  Held-out run accuracy 0.9583; macro F1 0.9580
Vibration result: matches the committed result


,accuracy,macro_precision,macro_recall,macro_f1
"Random forest, held-out C/D recordings (24)",0.958,0.964,0.958,0.958
Held-out speed profile (24),0.958,0.960,0.958,0.958
Shuffled training labels (control),0.083,0.077,0.083,0.080
Most-frequent class (control),0.250,0.062,0.250,0.100


![Vibration confusion matrix](../Figures/pump_bearing_independent_confusions.png)

## 2. Computer-vision branch: event camera with OpenCV

The first two seconds of each XJTU recording become eight 250 ms event-count images. OpenCV HOG features feed a random forest trained on the 24 recordings at 20 Hz and tested on the 24 separate recordings at 30 Hz. A harder check holds out whole camera configurations. These are motion images of the bearing housing, not photographs of visible damage.

When the two Zenodo zips are local, the script reads them from disk instead of the internet (same zip members, same code).

In [3]:
import zipfile
import remotezip

before = load("pump_event_visual.json")
if LOCAL["CV: XJTU-DV-Pump event camera (2 zips, 16.5 GB)"]:
    def local_zip(url, *args, **kwargs):
        part = "part1" if "part1" in url else "part2" if "part2" in url else None
        if part is None:
            raise ValueError(f"Unexpected archive URL: {url}")
        return zipfile.ZipFile(RAW / "xjtu_dv_pump" / f"XJTU-DV-Pump_{part}.zip")
    remotezip.RemoteZip = local_zip
    run_script("Models/pump_event_visual_baseline.py", "--refresh")  # rebuild features from the raw recordings
cv = load("pump_event_visual.json")
check(before, cv, [("random_forest_run_level",), ("held_out_camera_configuration_run_level",)], "Event-camera result")
display(pd.DataFrame({
    "Random forest, held-out 30 Hz recordings (24)": metrics(cv["random_forest_run_level"]),
    "Held-out camera configurations (48)": metrics(cv["held_out_camera_configuration_run_level"]),
    "Shuffled training labels (control)": metrics(cv["shuffled_training_run_labels_control"]),
    "Most-frequent class (control)": metrics(cv["dummy_run_level"]),
}).T)

[Models/pump_event_visual_baseline.py]
  30 Hz 24/24: outer 20,681,996 events
  Saved Results\pump_event_visual.json and Figures\pump_event_visual_confusion.png
  Run-level test macro F1: 1.0000
Event-camera result: matches the committed result


,accuracy,macro_precision,macro_recall,macro_f1
"Random forest, held-out 30 Hz recordings (24)",1.000,1.000,1.000,1.000
Held-out camera configurations (48),0.854,0.868,0.854,0.855
Shuffled training labels (control),0.208,0.196,0.208,0.202
Most-frequent class (control),0.250,0.062,0.250,0.100


![Event-camera confusion matrix](../Figures/pump_event_visual_confusion.png)

## 3. Sensor vs computer vision

In [4]:
run_script("Models/pump_compare_modalities.py", tail=1)
comparison = pd.DataFrame([
    {"branch": "Sensor: vibration", "test units": "24 C/D recordings", **metrics(sensor["random_forest_run_level"])},
    {"branch": "CV: event camera", "test units": "24 recordings at 30 Hz", **metrics(cv["random_forest_run_level"])},
    {"branch": "CV: held-out camera configuration", "test units": "48 recordings", **metrics(cv["held_out_camera_configuration_run_level"])},
    {"branch": "Most-frequent class (control)", "test units": "24 recordings", **metrics(cv["dummy_run_level"])},
]).set_index("branch")[["test units", "accuracy", "macro_precision", "macro_recall", "macro_f1"]]
display(comparison)

[Models/pump_compare_modalities.py]
  Saved Results\pump_sensor_visual_comparison.md and Figures\pump_sensor_visual_confusions.png


,test units,accuracy,macro_precision,macro_recall,macro_f1
branch,,,,,
Sensor: vibration,24 C/D recordings,0.958,0.964,0.958,0.958
CV: event camera,24 recordings at 30 Hz,1.000,1.000,1.000,1.000
CV: held-out camera configuration,48 recordings,0.854,0.868,0.854,0.855
Most-frequent class (control),24 recordings,0.250,0.062,0.250,0.100


![Pump sensor vs CV confusion matrices](../Figures/pump_sensor_visual_confusions.png)

**Interpretation.** Both branches separate the four bearing conditions well on held-out recordings (vibration 0.958, event camera 1.000), far above the 0.250 control. One vibration ball-fault run was predicted as inner race. Holding out whole camera configurations drops the event camera to about 0.85, so its 1.000 depends partly on seeing the same camera set-ups in training. The branches use different rigs and operating shifts, so this is not a controlled modality ranking. Full write-up: [`Results/pump_sensor_visual_comparison.md`](../Results/pump_sensor_visual_comparison.md).

## 4. Supporting sensor-only baselines

These are **sensor-only** pump results. They have no image branch and are **not** part of the sensor-vs-CV comparison above.

- **Original centrifugal-pump XLSX** (Mendeley 10.17632/63gjxpn7s6.1): one file per class, so train and test windows come from the same recording. Pipeline diagnostic only.
- **UCI hydraulic** (DOI 10.24432/C5CW21): internal hydraulic-pump leakage severity on a test rig, complete condition combinations held out.
- **ESPset V3** (DOI 10.17632/m268jsw339.3): offshore electric submersible pumps, normal vs abnormal vibration, pump IDs held out.

**Reproducibility note.** The committed UCI and ESPset results were made with package versions that were not recorded. Under scikit-learn 1.9.1 they give 0.963 (committed 0.970) and 0.914 (committed 0.918). The UCI split comes from `StratifiedGroupKFold(shuffle=True)`, which changes between scikit-learn versions, and matches exactly under 1.6.1.

In [5]:
supporting = [
    ("Supporting: original pump XLSX", "Models/pump_bearing_baseline.py", "pump_bearing_exploratory.json",
     "Original pump XLSX: 4 classes, same-recording windows (diagnostic)", lambda d: d["random_forest"]),
    ("Supporting: UCI hydraulic", "Preprocessing/hydraulic_baseline.py", "uci_hydraulic_pump_leakage.json",
     "UCI hydraulic: pump leakage, 3 classes", lambda d: d["test_selected"]),
    ("Supporting: ESPset V3", "Models/espset_baseline.py", "espset_v3_binary_fault.json",
     "ESPset V3: normal vs abnormal", lambda d: d["test_selected"]),
]
rows = {}
for key, script, name, label, pick in supporting:
    before = load(name)
    if LOCAL[key]:
        run_script(script, tail=1)
    after = load(name)
    rows[label] = {"committed macro F1": round(pick(before)["macro_f1"], 3),
                   "this run macro F1": round(pick(after)["macro_f1"], 3),
                   "this run accuracy": round(pick(after)["accuracy"], 3)}
display(pd.DataFrame(rows).T)

[Models/pump_bearing_baseline.py]
  RF macro F1 1.0000; scientifically valid independent-run score: unavailable


[Preprocessing/hydraulic_baseline.py]
  Wrote Results\uci_hydraulic_pump_leakage.json and Figures\uci_hydraulic_pump_leakage.png


[Models/espset_baseline.py]
  Wrote Results\espset_v3_binary_fault.json and Figures\espset_v3_binary_fault.png


,committed macro F1,this run macro F1,this run accuracy
"Original pump XLSX: 4 classes, same-recording windows (diagnostic)",1.000,1.000,1.000
"UCI hydraulic: pump leakage, 3 classes",0.970,0.963,0.962
ESPset V3: normal vs abnormal,0.918,0.914,0.937


## Conclusion

The Pump is the stronger matched-condition demonstration: a vibration model and an event-camera model each classify the same four present bearing conditions on recordings they never saw in training. Because the two branches come from different laboratory rigs, the result shows that both modalities can carry the fault signal, not which one is better on a real mining pump.

**Data citations (CC BY 4.0):** University of Ottawa bearing vibration V2, DOI 10.17632/v43hmbwxpm.2; XJTU-DV, Zenodo DOI 10.5281/zenodo.17447083. Supporting datasets are cited in [`Dataset_Manifest/manifest.md`](../Dataset_Manifest/manifest.md).